In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from huggingface_hub import InferenceClient

from dotenv import load_dotenv
import os

ModuleNotFoundError: No module named 'langgraph'

In [ ]:
load_dotenv()

api_key = os.getenv('HUGGINGFACE_API_KEY')
if not api_key:
    raise ValueError("HUGGINGFACE_API_KEY not found in .env file. Please add your API key.")
print(f"API key loaded: {api_key[:10]}...")


ValueError: HUGGINGFACE_API_KEY not found in .env file. Please add your API key.

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-7B-Instruct',
    huggingfacehub_api_token=api_key
) 

In [ ]:
model =ChatHuggingFace(llm=llm)

In [ ]:
# define state
class LLMstate(TypedDict):
    question : str
    answer : str

In [ ]:
# define graph
graph  = StateGraph(LLMstate)


In [ ]:
def Llm_call_qa(state :LLMstate )-> LLMstate:
    # extract question from state 
     quest = state['question']
    # preapred prompt and send to model
     prompt = f'provide the appropriate answer for this {quest}'
    # extarct ans 
     answer = model.invoke(prompt).content
# set answer to state
     state['answer'] = answer
     return state

In [ ]:
# define nodes 
if 'Llm_call_qa' not in graph.nodes:
    graph.add_node('Llm_call_qa', Llm_call_qa)

# define edge
graph.add_edge(START,'Llm_call_qa')
graph.add_edge('Llm_call_qa',END)

In [ ]:
# compile graph
workflow= graph.compile()
initial_state = {'question':'what is kidlins law or problem solving'}
final_state = workflow.invoke(initial_state)
print(final_state)
